# T04 — 근육과 EMG: 근육 역학, EMG 기록·처리·정규화, EMG–힘 관계의 한계

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait-theory/note-T04.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait-theory/note-T04.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 본문

### 8. 근골격 모델과 정적 최적화의 비유일성

아래 코드는 이 노트를 위해 새로 작성했습니다. ① twitch 시간과 envelope cutoff, ② Hill 곡선, ③ 가상의 굴곡근 셋에 대한 정적 최적화를 확인합니다.

In [ ]:
import numpy as np
from scipy.optimize import minimize

# 1) twitch를 2차 저역통과의 임펄스 응답으로 보면 envelope cutoff는 1/(2πT)
for T_ms in (40, 53, 80, 106):
    print(f"twitch 수축 시간 T={T_ms:3d} ms → f_c = {1/(2*np.pi*T_ms/1000):.2f} Hz")
t = np.linspace(0, 0.6, 6001); T = 0.08
F = (t/T)*np.exp(-t/T)
print("twitch 봉우리 시점(ms):", round(t[F.argmax()]*1000, 1), " 봉우리/F0:", round(F.max(), 3))

# 2) Hill 식 (P + a)(v + b) = (P0 + a)b, 단축 속도 v ≥ 0
P0, a_over_P0 = 1.0, 0.25          # 설명용으로 고른 가상의 a/P0
a = a_over_P0*P0; Vmax = 1.0       # 속도를 Vmax로 정규화
b = a*Vmax/P0                      # P=0일 때 v=Vmax가 되도록
for v in (0, 0.1, 0.25, 0.5, 1.0):
    P = (P0 + a)*b/(v + b) - a
    print(f"v/Vmax={v:4.2f} → P/P0={P:5.3f}")

# 3) 정적 최적화의 해는 비용 함수에 따라 바뀐다 (가상 굴곡근 3개)
r = np.array([0.04, 0.03, 0.05])   # 모멘트 팔 (m)
A = np.array([6.0, 4.0, 3.0])      # 단면적 (cm²)
M = 20.0                           # 필요한 관절 모멘트 (N·m)
for p in (1, 2, 3):
    res = minimize(lambda f: np.sum((f/A)**p), x0=np.full(3, 100.0), method="SLSQP",
                   bounds=[(0, None)]*3, constraints={"type": "eq", "fun": lambda f: r @ f - M})
    print(f"p={p}: 근력(N) =", np.round(res.x, 1), " 모멘트 확인:", round(r @ res.x, 2))
for p in (2, 3):
    k = (r*A**p)**(1/(p-1)); f = k*M/(r @ k)
    print(f"p={p} 닫힌 해:", np.round(f, 1))